# **Conduct Power Analysis**

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

def calculate_sample_size(p0, delta, alpha=0.05, power=0.80):
    z_alpha = stats.norm.ppf(1-alpha/2)
    z_beta = stats.norm.ppf(power)

    p1 = p0 * (1-delta)
    pooled_prob = (p0 + p1)/2

    n = (2 * (z_alpha + z_beta)**2 * pooled_prob * (1 - pooled_prob)) / (delta * p0) ** 2
    return n

def calculate_mlds(p0, daily_samples, weeks, alpha=0.05, power=0.80):
    lifts = []
    for week in weeks:
        days_needed = week * 7
        total_samples = days_needed * daily_samples
        sample_size_per_group = total_samples / 2

        delta = 0.0001
        step = 0.0001
        while True:
            needed_samples = calculate_sample_size(p0, delta, alpha, power)
            if needed_samples <= sample_size_per_group:
                lifts.append(delta * 100)
                break
            delta += step
    return lifts

# Sample parameters
baseline_conversion = 0.04  # Example: 0.10 for 10% baseline conversion rate
daily_samples = 0.10*4000  # Daily Samples
weeks = np.arange(1, 11)  # Weeks for Visualization
mdls = calculate_mlds(baseline_conversion, daily_samples, weeks)

df = pd.DataFrame({'Weeks': weeks, 'Lift': mdls})

# **Visualize Power Analysis using Plotly.js**

In [3]:
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Bar(x=df.Weeks, y=df.Lift, marker_color='#be2557'),
    ],
    layout=dict(
        barcornerradius=56,
    ),
)

fig.update_layout(
    title='How many weeks to run a test?',
    autosize=False,
    width=800,
    height=500,
    xaxis_tickfont_size=14,
    yaxis=dict(
        title='Minimal Detectable Effect (%)',
        title_font=dict(size=16),  # Changed here
        tickfont_size=14,
    ),
    xaxis=dict(
        title='Test Duration (Weeks)',
        tickvals=weeks,
        title_font=dict(size=16),  # Changed here
        tickfont_size=14,
    ),
    legend=dict(
        x=0,
        y=1.0,
        bgcolor='rgba(255, 255, 255, 0)',
        bordercolor='rgba(255, 255, 255, 0)'
    ),
    barmode='group',
    bargap=0.15,  # gap between bars of adjacent location coordinates.
    bargroupgap=0.15  # gap between bars of the same location coordinate.
)

fig.show()
